In [9]:
# library installation
%pip install insightface
%pip install retina-face
%pip install pyiqa
%pip install pandas
%pip install scikit-learn
%pip install tqdm


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
# imports
# imports
import os
import cv2
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import pyiqa

from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity

from retinaface import RetinaFace

from insightface.app import FaceAnalysis

In [11]:
# paths

ROOT_DIR = "celebrity_dataset"

GALLERY_DIR = os.path.join(
    ROOT_DIR,
    "gallery",
)

PROBE_DIR = os.path.join(
    ROOT_DIR,
    "probe",
)

GALLERY_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "gallery_embeddings",
)

PROBE_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "probe_embeddings",
)

OUTPUT_DATASET = "celebrity_dataset.csv"

In [12]:
# creating output directories

os.makedirs(
    GALLERY_DIR,
    exist_ok=True,
)

os.makedirs(
    GALLERY_EMBED_DIR,
    exist_ok=True,
)

os.makedirs(
    PROBE_EMBED_DIR,
    exist_ok=True,
)

print("Directories Ready")

Directories Ready


In [16]:
# intializing arcface

app = FaceAnalysis(
    name="buffalo_l",
)

app.prepare(
    ctx_id=0 if torch.cuda.is_available() else -1,
    det_size=(640,640),
)

rec_model = app.models["recognition"]

print("Recognition Model Loaded")
print("ArcFace Loaded")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
se

In [17]:
# --------------------------
# ArcFace Embedding Function
# --------------------------


def get_embedding(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.resize(img, (112, 112))

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    emb = rec_model.get_feat(img).flatten()

    return emb.astype(np.float32)

In [18]:
# --------------------------
# Gallery Embeddings
# --------------------------

saved = 0

failed = 0

for identity in tqdm(sorted(os.listdir(GALLERY_DIR))):

    identity_dir = os.path.join(
        GALLERY_DIR,
        identity,
    )

    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(
        GALLERY_EMBED_DIR,
        identity,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    for image_name in sorted(os.listdir(identity_dir)):

        image_path = os.path.join(
            identity_dir,
            image_name,
        )

        embedding = get_embedding(
            image_path,
        )

        if embedding is None:

            failed += 1
            continue

        save_path = os.path.join(
            save_dir,
            os.path.splitext(image_name)[0] + ".npy",
        )

        np.save(
            save_path,
            embedding,
        )

        saved += 1

print()

print("Gallery Embeddings Saved :", saved)

print("Failures :", failed)

100%|██████████| 100/100 [00:10<00:00,  9.90it/s]


Gallery Embeddings Saved : 100
Failures : 0


In [19]:
# --------------------------
# Probe Embeddings
# --------------------------

saved = 0

failed = 0

for identity in tqdm(sorted(os.listdir(PROBE_DIR))):

    identity_dir = os.path.join(
        PROBE_DIR,
        identity,
    )

    if not os.path.isdir(identity_dir):
        continue

    save_dir = os.path.join(
        PROBE_EMBED_DIR,
        identity,
    )

    os.makedirs(
        save_dir,
        exist_ok=True,
    )

    for image_name in sorted(os.listdir(identity_dir)):

        image_path = os.path.join(
            identity_dir,
            image_name,
        )

        embedding = get_embedding(
            image_path,
        )

        if embedding is None:

            failed += 1
            continue

        save_path = os.path.join(
            save_dir,
            os.path.splitext(image_name)[0] + ".npy",
        )

        np.save(
            save_path,
            embedding,
        )

        saved += 1

print()

print("Probe Embeddings Saved :", saved)

print("Failures :", failed)

100%|██████████| 100/100 [00:10<00:00,  9.42it/s]


Probe Embeddings Saved : 100
Failures : 0


In [20]:
# loading MUSIQ

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

musiq_metric = pyiqa.create_metric(
    "musiq",
    device=device,
)

print("MUSIQ Loaded")

Device : cpu
Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth
MUSIQ Loaded


In [21]:
# -------------------------
# MUSIQ Quality
# -------------------------


def get_quality_score(image_path):

    score = musiq_metric(
        image_path,
    )

    return float(score.cpu().item())

In [22]:
# -------------------------
# Gallery Loader
# -------------------------


def load_gallery_database(
    gallery_root,
):

    gallery_db = {}

    total = 0

    for identity in sorted(os.listdir(gallery_root)):

        identity_dir = os.path.join(
            gallery_root,
            identity,
        )

        if not os.path.isdir(identity_dir):
            continue

        gallery_db[identity] = []

        for emb_file in sorted(os.listdir(identity_dir)):

            if not emb_file.endswith(".npy"):
                continue

            emb = np.load(
                os.path.join(
                    identity_dir,
                    emb_file,
                )
            )

            gallery_db[identity].append(
                emb,
            )

            total += 1

    print("Gallery Identities :", len(gallery_db))

    print("Gallery Embeddings :", total)

    return gallery_db

In [23]:
# -------------------------
# Probe Loader
# -------------------------


def load_probe_database(
    probe_root,
):

    probe_db = {}

    total = 0

    for identity in sorted(os.listdir(probe_root)):

        identity_dir = os.path.join(
            probe_root,
            identity,
        )

        if not os.path.isdir(identity_dir):
            continue

        files = [f for f in os.listdir(identity_dir) if f.endswith(".npy")]

        if len(files) == 0:
            continue

        emb = np.load(
            os.path.join(
                identity_dir,
                files[0],
            )
        )

        probe_db[identity] = emb

        total += 1

    print("Probe Embeddings :", total)

    return probe_db

In [24]:
# loading gallery and probe embeddings

gallery_db = load_gallery_database(
    GALLERY_EMBED_DIR,
)

probe_db = load_probe_database(
    PROBE_EMBED_DIR,
)

Gallery Identities : 100
Gallery Embeddings : 100
Probe Embeddings : 100


In [25]:
# -------------------------
# Gallery Search
# -------------------------


def search_gallery(
    probe_embedding,
    gallery_db,
):

    identity_scores = {}

    for identity in gallery_db:

        similarities = []

        for gallery_embedding in gallery_db[identity]:

            similarity = cosine_similarity(
                probe_embedding.reshape(
                    1,
                    -1,
                ),
                gallery_embedding.reshape(
                    1,
                    -1,
                ),
            )[0][0]

            similarities.append(
                similarity,
            )

        identity_scores[identity] = max(
            similarities,
        )

    sorted_scores = sorted(
        identity_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    return (
        identity_scores,
        sorted_scores,
    )

In [26]:
# -------------------------
# Dataset Generation
# -------------------------

rows = []

for identity in tqdm(probe_db):

    probe_embedding = probe_db[identity]

    probe_folder = os.path.join(
        PROBE_DIR,
        identity,
    )

    image_name = None

    for file in os.listdir(probe_folder):

        if file.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
            )
        ):

            image_name = file
            break

    if image_name is None:
        continue

    image_path = os.path.join(
        probe_folder,
        image_name,
    )

    quality_score = get_quality_score(
        image_path,
    )

    identity_scores, sorted_scores = search_gallery(
        probe_embedding,
        gallery_db,
    )

    ##################################################
    # Genuine Pair
    ##################################################

    genuine_similarity = identity_scores[identity]

    best_impostor_similarity = max(
        score for gallery_id, score in identity_scores.items() if gallery_id != identity
    )

    genuine_margin = genuine_similarity - best_impostor_similarity

    rows.append(
        {
            "quality_score": quality_score,
            "best_similarity": genuine_similarity,
            "margin": genuine_margin,
            "label": 1,
            "true_identity": identity,
            "gallery_identity": identity,
        }
    )

    ##################################################
    # Hard Negative
    ##################################################

    impostor_scores = [
        (gallery_id, similarity)
        for gallery_id, similarity in sorted_scores
        if gallery_id != identity
    ]

    hard_negative_identity = impostor_scores[0][0]

    hard_negative_similarity = impostor_scores[0][1]

    second_impostor_similarity = impostor_scores[1][1]

    hard_negative_margin = hard_negative_similarity - second_impostor_similarity

    rows.append(
        {
            "quality_score": quality_score,
            "best_similarity": hard_negative_similarity,
            "margin": hard_negative_margin,
            "label": 0,
            "true_identity": identity,
            "gallery_identity": hard_negative_identity,
        }
    )

100%|██████████| 100/100 [00:14<00:00,  6.79it/s]


In [27]:
# saving dataset
dataset = pd.DataFrame(
    rows,
)

dataset.to_csv(
    OUTPUT_DATASET,
    index=False,
)

print()

print("Dataset Saved")

print()

print(dataset.shape)

print()

display(dataset.head())


Dataset Saved

(200, 6)



,quality_score,best_similarity,margin,label,true_identity,gallery_identity
0,17.637470,0.550108,0.374149,1,Alice_Krige,Alice_Krige
1,17.637470,0.175959,0.005657,0,Alice_Krige,Joshua_Jackson
2,19.537975,0.351924,0.187498,1,Alyssa_Milano,Alyssa_Milano
3,19.537975,0.164426,0.022867,0,Alyssa_Milano,Gates_McFadden
4,22.501440,0.751467,0.618040,1,Amaury_Nolasco,Amaury_Nolasco


In [28]:
# dataset stats
print()

print("Label Counts")

print(dataset["label"].value_counts())

print()

print("Quality Statistics")

print(dataset["quality_score"].describe())

print()

print("Similarity Statistics")

print(dataset["best_similarity"].describe())

print()

print("Margin Statistics")

print(dataset["margin"].describe())


Label Counts
label
1    100
0    100
Name: count, dtype: int64

Quality Statistics
count    200.000000
mean      20.962221
std        2.457069
min       15.659685
25%       19.367360
50%       20.499352
75%       22.719759
max       26.945877
Name: quality_score, dtype: float64

Similarity Statistics
count    200.000000
mean       0.279617
std        0.144405
min       -0.027753
25%        0.165137
50%        0.215180
75%        0.383425
max        0.751467
Name: best_similarity, dtype: float64

Margin Statistics
count    200.000000
mean       0.121734
std        0.134604
min       -0.166733
25%        0.013649
50%        0.058100
75%        0.224874
max        0.618040
Name: margin, dtype: float64


In [29]:
# positive vs negative stats
print("Similarity")

print(dataset.groupby("label")["best_similarity"].describe())

print()

print("Margin")

print(dataset.groupby("label")["margin"].describe())

Similarity
       count      mean       std       min       25%       50%       75%  \
label                                                                      
0      100.0  0.170134  0.033658  0.102145  0.145794  0.165930  0.189589   
1      100.0  0.389100  0.128716 -0.027753  0.299714  0.383637  0.471106   

            max  
label            
0      0.286501  
1      0.751467  

Margin
       count      mean       std       min       25%       50%       75%  \
label                                                                      
0      100.0  0.024502  0.024757  0.000059  0.008019  0.015495  0.035333   
1      100.0  0.218967  0.129256 -0.166733  0.130492  0.225920  0.302683   

            max  
label            
0      0.140952  
1      0.618040  
